# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

> **Current stage: Phase 2 — Temporal Reconstruction Audit.**

Phase 1 is preserved in Git history. This version keeps the current reproducible corpus backbone and tests the two remaining issues before any semantic network is built: **Herrera/Pacheco reconciliation** and the **actual temporal evidence available for each poem/source**.

## Colab ↔ GitHub workflow

1. Open this notebook from `ardominguezm/golden-age-semantic-reconfiguration`.
2. Run **Runtime → Run all**.
3. Inspect the final checkpoint.
4. Save the executed notebook back to this same GitHub path on `main`.

We still do **not** choose temporal windows or build semantic networks.

## What Phase 1 established

- Navarro TEI: **5,078** poem-level records, 53 author folders.
- Hernández-Lorenzo: **4,381** recovered blank-line blocks.
- **4,065 (92.8%)** link exactly to Navarro after normalized-text matching; another 194 are near-identical at ≥98%.
- Hernández `Date` is author lifespan, not poem chronology.
- Explicit TEI dates are extremely sparse and may be witness/edition dates.
- Navarro is therefore the poem-identity backbone; Hernández is a standardized textual layer plus source-exclusive material such as Pacheco.

In [ ]:
import sys, re, shutil, subprocess, unicodedata
from pathlib import Path
from collections import defaultdict
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES={
 'hernandez_network':('https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git','ef6b7b691f67abe60d9cfa85c274f0be8095dd9a'),
 'navarro_tei':('https://github.com/bncolorado/CorpusSonetosSigloDeOro.git','092a5fe70a4065a4d84bfed288bffd3851348f9c')}
ROOT=Path('/content/gasr_sources'); ROOT.mkdir(exist_ok=True)

def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(['git','clone','--quiet',url,str(dst)],check=True)
    subprocess.run(['git','-C',str(dst),'checkout','--quiet',commit],check=True)
    got=subprocess.check_output(['git','-C',str(dst),'rev-parse','HEAD'],text=True).strip()
    assert got==commit
    return dst

paths={k:clone(k,*v) for k,v in SOURCES.items()}
H=paths['hernandez_network']; N=paths['navarro_tei']
NS={'tei':'http://www.tei-c.org/ns/1.0'}

def txt(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize('NFKD',str(s)); s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]','',s.lower())
def years(s): return sorted(set(re.findall(r'(?<!\d)(1[45-7]\d{2})(?!\d)',str(s))))
def blocks(s):
    out=[]; cur=[]
    for line in s.splitlines():
        if line.strip(): cur.append(line.strip())
        elif cur: out.append(cur); cur=[]
    if cur: out.append(cur)
    return out

print('Pinned sources ready')
print('Python',sys.version.split()[0],'| pandas',pd.__version__)

## 01. Rebuild Navarro and separate temporal evidence channels

Dates in a textual witness or edition are not composition dates. Years embedded in a bibliographic description are also not automatically poem dates. The notebook therefore preserves these channels separately and **never creates `composition_year` from them**.

In [ ]:
rows=[]
for p in sorted(N.rglob('*.xml')):
    root=ET.parse(p).getroot()
    ls=[txt(x) for x in root.findall('.//tei:l',NS)]; ls=[x for x in ls if x]
    title=txt(root.find('.//tei:body/tei:head/tei:title',NS))
    bibl=txt(root.find('.//tei:sourceDesc/tei:bibl',NS))
    wtitles=[]; wyears=[]
    for w in root.findall('.//tei:sourceDesc//tei:witness',NS):
        wtitles.append(txt(w.find('tei:title',NS)))
        for d in w.findall('.//tei:date',NS): wyears += years(txt(d))
    alld=[]
    for d in root.findall('.//tei:date',NS): alld += years(txt(d))
    other=sorted(set(alld)-set(wyears))
    rows.append(dict(n_id=str(p.relative_to(N)).replace('/','::'),author_dir=p.parent.name,title=title,
        text_tei='\n'.join(ls),n_lines=len(ls),source_file=str(p.relative_to(N)),source_bibl=bibl,
        source_bibl_years=';'.join(years(bibl)),witness_titles=' || '.join(x for x in wtitles if x),
        witness_years=';'.join(sorted(set(wyears))),other_tei_years=';'.join(other)))

n=pd.DataFrame(rows); n['signature']=n.text_tei.map(norm)
print(f'Navarro poems: {len(n):,} | authors: {n.author_dir.nunique()} | 14-line: {(n.n_lines==14).sum():,}')

date_audit=pd.DataFrame({
 'channel':['witness/edition year','other explicit TEI date year','year string in source bibliography'],
 'files':[n.witness_years.ne('').sum(),n.other_tei_years.ne('').sum(),n.source_bibl_years.ne('').sum()]})
display(date_audit)
display(n[(n.witness_years!='')|(n.other_tei_years!='')|(n.source_bibl_years!='')][['n_id','author_dir','title','source_bibl','source_bibl_years','witness_titles','witness_years','other_tei_years']].head(30))
print('No field above is being promoted to composition_year.')

## 02. Source / collection concentration

If the 5,078 sonnets reduce to a manageable number of repeated author–source groups, the historical reconstruction can be organized by **source/collection and author** rather than as 5,078 isolated searches.

In [ ]:
source_groups=(n.groupby(['author_dir','source_bibl'],dropna=False).agg(
 poems=('n_id','count'),witness_dated_files=('witness_years',lambda s:int(s.ne('').sum())),
 source_year_files=('source_bibl_years',lambda s:int(s.ne('').sum()))).reset_index()
 .sort_values(['poems','author_dir'],ascending=[False,True]))

author_sources=(source_groups.groupby('author_dir').agg(
 n_poems=('poems','sum'),n_source_descriptions=('source_bibl','nunique'),
 files_with_witness_dates=('witness_dated_files','sum'),files_with_source_years=('source_year_files','sum'))
 .reset_index().sort_values('n_poems',ascending=False))

print('Distinct author + source-description groups:',len(source_groups))
display(source_groups.head(40))
display(author_sources.head(40))

## 03. Rebuild Hernández exact links

For this phase we use **exact normalized-text evidence only** to infer author correspondences. This also avoids the diagnostic bug in the previous author-specific `unresolved` column, which counted exact matches as unresolved because exact matches naturally had no fuzzy score.

In [ ]:
hrows=[]
for p in sorted(set((H/'corpus').glob('*Sonetos*.txt'))):
    a=re.sub(r'_Sonetos.*$','',p.stem)
    for i,b in enumerate(blocks(p.read_text(encoding='utf-8',errors='replace')),1):
        t='\n'.join(b); hrows.append(dict(h_id=f'{a}_{i:04d}',author_file=a,source_file_h=p.name,n_lines_h=len(b),text_standardized=t,signature=norm(t)))
h=pd.DataFrame(hrows)
idx=defaultdict(list)
for r in n[['n_id','author_dir','signature']].itertuples(index=False): idx[r.signature].append((r.n_id,r.author_dir))
def exact(sig):
    z=idx.get(sig,[]); return z[0] if len(z)==1 else (pd.NA,pd.NA)
z=h.signature.map(exact); h['exact_n_id']=[x[0] for x in z]; h['exact_author_dir']=[x[1] for x in z]; h['is_exact']=h.exact_n_id.notna()
mapbest=(h[h.is_exact].groupby(['author_file','exact_author_dir']).size().reset_index(name='exact_poems')
 .sort_values(['author_file','exact_poems'],ascending=[True,False]).groupby('author_file',as_index=False).head(1))
print(f'Hernández blocks: {len(h):,} | exact Navarro links: {h.is_exact.sum():,} ({h.is_exact.mean():.1%})')
display(mapbest)

## 04. Herrera / `AN` / Pacheco

Both `AN` and `Herrera` previously mapped to `FernandoDeHerrera`. We now measure whether they are duplicate, overlapping, or largely distinct textual layers. This is essential because duplicating Herrera would bias any later semantic network.

In [ ]:
def sigset(a): return set(h.loc[h.author_file.eq(a),'signature'])
an,he,pa=sigset('AN'),sigset('Herrera'),sigset('Pacheco')
hnav=set(n.loc[n.author_dir.eq('FernandoDeHerrera'),'signature'])
special=pd.DataFrame([
 {'layer':'AN','blocks':len(an),'overlap_AN_Herrera':len(an&he),'exact_overlap_Navarro_Herrera':len(an&hnav)},
 {'layer':'Herrera','blocks':len(he),'overlap_AN_Herrera':len(an&he),'exact_overlap_Navarro_Herrera':len(he&hnav)},
 {'layer':'Pacheco','blocks':len(pa),'overlap_AN_Herrera':0,'exact_overlap_Navarro_Herrera':len(pa&hnav)}])
display(special)

hlinks=h[h.is_exact & h.exact_author_dir.eq('FernandoDeHerrera')][['h_id','author_file','exact_n_id']]
coll=(hlinks.groupby('exact_n_id').agg(n_h_links=('h_id','count'),layers=('author_file',lambda s:', '.join(sorted(set(s))))).reset_index())
coll=coll[coll.n_h_links>1]
print('Cross-layer exact collisions on Navarro Herrera:',len(coll))
display(coll.head(30))
print('Pacheco blocks in Hernández:',int(h.author_file.eq('Pacheco').sum()))
print('Pacheco author folder present in Navarro:',bool(n.author_dir.str.contains('Pacheco',case=False,regex=False).any()))

## 05. Temporal Reconstruction Worklist

This is the main operational output. `research_priority` is a workflow priority for the current Renaissance–Baroque question, **not a literary ranking**. The worklist deliberately asks first for critical-edition composition intervals and only then for first-publication/circulation evidence.

In [ ]:
central={'GarcilasoDeLaVega','JuanBoscan','FernandoDeHerrera','PedroEspinosa','JuanDeArguijo','JuanDeJauregui','LuisCarrilloYSotomayor','Cervantes','Gongora','LopeDeVega_1','LopeDeVega_2','Quevedo'}
work=author_sources.copy()
work['temporal_status']=work.apply(lambda r:('witness/edition evidence only — external dating still required' if r.files_with_witness_dates>0 else ('bibliographic year string only — external dating required' if r.files_with_source_years>0 else 'no explicit poem-level temporal evidence — external dating required')),axis=1)
work['research_priority']=work.apply(lambda r:'A — central transition author' if r.author_dir in central else ('B — high corpus weight' if r.n_poems>=100 else 'C — standard'),axis=1)
work['next_evidence_target']='critical edition / scholarly chronology: composition interval; if unavailable, first publication or collection circulation'
order={'A — central transition author':0,'B — high corpus weight':1,'C — standard':2}
work['_o']=work.research_priority.map(order); work=work.sort_values(['_o','n_poems'],ascending=[True,False]).drop(columns='_o')
display(work)
print('\nPriority counts:'); display(work.research_priority.value_counts().rename_axis('priority').reset_index(name='authors'))
print('\nExplicit temporal coverage on TEI backbone')
print('Total poems:',len(n))
print('Files with witness years:',int(n.witness_years.ne('').sum()))
print('Files with other TEI years:',int(n.other_tei_years.ne('').sum()))
print('Files with year strings in source bibliography:',int(n.source_bibl_years.ne('').sum()))

## 06. Runtime exports

These CSVs are convenience outputs inside the current Colab runtime. The executed notebook remains the preserved audit record until the derived-table schema is stable.

In [ ]:
OUT=Path('/content/gasr_phase2_outputs'); OUT.mkdir(exist_ok=True)
work.to_csv(OUT/'temporal_reconstruction_worklist.csv',index=False)
source_groups.to_csv(OUT/'source_groups.csv',index=False)
special.to_csv(OUT/'herrera_pacheco_diagnostics.csv',index=False)
print('Runtime outputs:')
for p in sorted(OUT.glob('*.csv')): print(' -',p)

## Scientific checkpoint

After execution, save this notebook back to GitHub. The outputs will determine:

1. whether `AN` and `Herrera` are disjoint, overlapping, or variant textual sets;
2. how concentrated the TEI corpus is by source/collection;
3. the genuine internal temporal coverage;
4. which authors/sources require external critical scholarship first;
5. whether Paper 1 can support fixed/adaptive windows or instead needs interval-weighted/probabilistic temporal assignment.

In [ ]:
print('PHASE 2 CHECKPOINT')
print('------------------')
print('Save this executed notebook to GitHub.')
print('Do NOT build semantic networks yet.')
print('Next: interpret source concentration + Herrera overlap + temporal worklist.')